<div align="center">

# Week 12: Digital Twins I: Synchronisation and Calibration

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-12) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/Week12_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also builds three weeks of the library B2 of the campus as the DataFrame `HIST`, simulated minute by minute and logged every 15 minutes. The values of the first twin, the library of Week 2, are kept in `FIRST`. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import least_squares

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The library B2 of the campus as it really behaves: the air and the furniture, the mass of the walls, the envelope and
# the gaps of the windows (a synthetic history, built from the model of Week 2 with a second heat capacity)
C_AIR, C_MASS = 20.0, 10.0          # kWh/K
R_AIR_MASS, R_MASS_OUT = 0.02, 0.46  # K/kW: from the air into the walls, from the walls to the outside
R_GAPS, R_WINDOW = 6.0, 0.7          # K/kW: the gaps of the closed windows, one window wide open
OPEN_FROM, OPEN_TO = pd.Timestamp("2026-02-18 10:00"), pd.Timestamp("2026-02-19 16:00")

def library_history(seed=12):
    """Three weeks of the library from Monday 2 February 2026, simulated minute by minute and logged every 15 minutes:
    the indoor sensor (noise 0.1 K), the outdoor temperature, the mean heater power, the sun on the windows and the
    opening hours. The thermostat holds 21 C from 6:00 (8:00 at weekends) to closing time and 17 C at night."""
    rng = np.random.default_rng(seed)
    days, n = 21, 21 * 24 * 60
    t = np.arange(n) / 60.0
    hour, day = t % 24, (t // 24).astype(int)
    weekday = day % 7
    level = np.zeros(days)
    for d in range(1, days):
        level[d] = 0.6 * level[d - 1] + rng.normal(0, 2.0)          # the weather of each day
    t_out = 3 + 4 * np.sin(2 * np.pi * (hour - 9) / 24) + np.interp(t, np.arange(days) * 24 + 12, level)
    cloud = rng.uniform(0.25, 1.0, days)
    sun = 450 * np.maximum(0, np.sin(np.pi * (hour - 7.5) / 9.5)) * cloud[day]
    is_open = np.where(weekday < 5, (hour >= 8) & (hour < 22), (hour >= 10) & (hour < 18))
    heated = np.where(weekday < 5, (hour >= 6) & (hour < 22), (hour >= 8) & (hour < 18))
    visitors = rng.uniform(0.6, 1.4, days)                          # people, lights and computers: about 6 kW
    gains = 6.0 * is_open * visitors[day]
    r_gaps = np.full(n, R_GAPS)
    r_gaps[(t >= (OPEN_FROM - pd.Timestamp("2026-02-02")) / pd.Timedelta(hours=1))
           & (t < (OPEN_TO - pd.Timestamp("2026-02-02")) / pd.Timedelta(hours=1))] = R_WINDOW
    air, mass, heat = np.zeros(n), np.zeros(n), np.zeros(n)
    air[0] = mass[0] = 18.0
    on = False
    for k in range(n - 1):
        setpoint = 21.0 if heated[k] else 17.0
        if air[k] < setpoint - 0.5:
            on = True
        elif air[k] > setpoint + 0.5:
            on = False
        heat[k] = 60.0 if on else 0.0
        q_air = (mass[k] - air[k]) / R_AIR_MASS + (t_out[k] - air[k]) / r_gaps[k] + heat[k] + 0.01 * sun[k] + gains[k]
        q_mass = (air[k] - mass[k]) / R_AIR_MASS + (t_out[k] - mass[k]) / R_MASS_OUT + 0.01 * sun[k]
        air[k + 1] = air[k] + q_air / C_AIR / 60
        mass[k + 1] = mass[k] + q_mass / C_MASS / 60
    heat[-1] = heat[-2]
    mean15 = lambda a: a.reshape(-1, 15).mean(axis=1)
    index = pd.date_range("2026-02-02", periods=n // 15, freq="15min")
    true = air[::15]
    return pd.DataFrame({"t_true": true.round(3), "t_in": (true + rng.normal(0, 0.1, len(true))).round(2),
                         "t_out": mean15(t_out).round(2), "heater_kw": mean15(heat).round(2),
                         "solar_wm2": mean15(sun).round(1), "open": mean15(is_open.astype(float))}, index=index)

HIST = library_history()
WEEK1, WEEK2, WEEK3 = HIST.loc["2026-02-02":"2026-02-08"], HIST.loc["2026-02-09":"2026-02-15"], HIST.loc["2026-02-16":]
DT = 0.25                                                           # hours between two log entries
FIRST = dict(C=30.0, R=0.4, g=0.0, q=0.0)                           # the first twin: the library of Week 2
print("Setup complete. Running in Colab:", IN_COLAB, "|", len(HIST), "entries from", HIST.index[0], "to", HIST.index[-1])

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 12.1": (lambda: (lambda r: [r.step(0.0, 40.0) for _ in range(4)][-1])(Room(25.0, 0.5, temp=18.0)), 1e-6),
    "Exercise 12.2": (lambda: least_squares(error, x0=[20.0, 0.6]).x[0], 1e-4),
    "Exercise 12.3": (lambda: 20.5, 1e-9),
    "Exercise A": (lambda: rmse(simulate(WEEK2, **FIRST)[0], WEEK2.t_in), 1e-6),
    "Exercise B": (lambda: rmse(simulate(WEEK2, **FIT)[0], WEEK2.t_in), 1e-6),
    "Exercise C": (lambda: rmse(simulate(WEEK2, **FIT, gain=0.2)[0], WEEK2.t_true), 1e-6),
    "Exercise D": (lambda: round(TRACKS[24].loc['2026-02-19 08:00'], 3), 1e-6),
    "Exercise E": (lambda: STAGE12['open window: first alarm after the opening (min)'], 1e-9),
}

## Python step 12: A twin as a class

A digital twin keeps a model and its state together and offers operations on them. This step writes a room as a Python class, fits its parameters with SciPy and adds a method that corrects the state with a measurement &#91;[4](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-4), [3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-3)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "HIST" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### A class that holds a model

In [ ]:
# Python practice: a small example for learning the language
class Room:
    """A room with one heat capacity C (kWh/K) and one thermal resistance R (K/kW) to the outside."""
    def __init__(self, C, R, temp=20.0):
        self.C, self.R, self.temp = C, R, temp           # the parameters and the state of this room
    def step(self, t_out, heat_kw, dt_h=0.25):
        """One Euler step of dt_h hours (Week 2)."""
        self.temp += dt_h / self.C * ((t_out - self.temp) / self.R + heat_kw)
        return self.temp

room = Room(30.0, 0.4, temp=17.0)
temps = [round(room.step(3.0, 60.0), 3) for _ in range(8)]   # two hours with 60 kW at 3 C outside
print("after each quarter of an hour:", temps)

A class bundles data with the functions that work on them, which are called methods. The method `__init__` runs when an object is created and stores its attributes in `self`. Each object of the class `Room` keeps its own parameters and its own temperature, so a program can run several rooms side by side.

### Fitting the parameters

In [ ]:
# Python practice: a small example for learning the language
measured = np.array([17.0, 17.26, 17.51, 17.76, 18.0, 18.24, 18.47, 18.7, 18.93])   # every 15 minutes, 60 kW on

def error(params):
    """The simulated temperatures of a room with the parameters C and R minus the measured ones."""
    trial = Room(*params, temp=measured[0])
    return np.array([measured[0]] + [trial.step(3.0, 60.0) for _ in range(8)]) - measured

fit = least_squares(error, x0=[20.0, 0.6])
print("fitted C and R:", fit.x.round(3), "| largest error:", round(float(np.abs(error(fit.x)).max()), 3), "K")

The function `least_squares` from SciPy changes the parameters until the sum of the squared errors is as small as it gets. The function `error` builds a new room for every trial, runs it with the same input as the measurement and returns the differences.

### A twin that follows the measurements

In [ ]:
# Python practice: a small example for learning the language
class TwinRoom(Room):
    """A room that also corrects its state with each measurement: a synchronised twin."""
    def sync(self, measured, gain):
        self.temp += gain * (measured - self.temp)
        return self.temp

twin = TwinRoom(30.0, 0.4, temp=19.0)                 # it starts 2 K too low
for reading in [21.0, 21.1, 21.0, 20.9]:
    twin.step(3.0, 50.0)
    print("twin after the update:", round(twin.sync(reading, 0.3), 3))

The class `TwinRoom(Room)` inherits every method of `Room` and adds `sync`. The gain decides how far the twin moves towards each measurement: 0 ignores it, 1 copies it.

**Quick check.** What is the difference between the class `Room` and the object `room`?

<details><summary>Answer</summary>

The class is the plan: It says which attributes and methods a room has. The object is one room built from the plan, with its own values of C, R and the temperature.

</details>

### Exercises for Python step 12

**Exercise 12.1.** Create a `Room` with C = 25 kWh/K, R = 0.5 K/kW and 18 °C. Run four steps at 0 °C outside with 40 kW of heat and store the final temperature in `t_121`.

In [ ]:
t_121 = None   # your answer

check("Exercise 12.1", t_121)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r = Room(25.0, 0.5, temp=18.0)
for _ in range(4):
    r.step(0.0, 40.0)
t_121 = r.temp

check("Exercise 12.1", t_121)

**Exercise 12.2.** Store the fitted heat capacity C of the example above in `c_122`.

In [ ]:
c_122 = None   # your answer

check("Exercise 12.2", c_122)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
c_122 = fit.x[0]

check("Exercise 12.2", c_122)

**Exercise 12.3.** A `TwinRoom` at 20.0 °C receives the measurement 21.0 °C with the gain 0.5. Store its temperature after `sync`, without a step, in `t_123`.

In [ ]:
t_123 = None   # your answer

check("Exercise 12.3", t_123)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_123 = TwinRoom(30.0, 0.4, temp=20.0).sync(21.0, 0.5)

check("Exercise 12.3", t_123)

## 1. The library and its first twin &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-2)&#93;

*Lecture: [What a digital twin is](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#digital-twin)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "TwinRoom" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The building management of the library logs five values every 15 minutes. They are the indoor and the outdoor temperature, the power of the heater, the sun on the windows and the opening hours. Python runs a first twin, the library of Week 2 with only the heater and the outdoor temperature, and compares it with three weeks of data.

**From the formulas to the code.** The twin is the model of Week 2 with two more heat flows: $C\,\dot{T} = (T_{out} - T)/R + P + g\,G + q\,o$. Here $G$ is the sun in W/m², $o$ the opening hours and $q$ the internal gains. One Euler step of $\Delta t = 0.25$ h is `simulate`, which also returns the prediction before each update. The root mean square error (RMSE) of Week 10 is `rmse`.

In [ ]:
def simulate(d, C, R, g=0.0, q=0.0, gain=0.0):
    """The twin over the rows of d: open loop with gain 0, synchronised with the measurement otherwise.
    Returns the twin temperature after each update and the one-step prediction before it."""
    t_out, heat, sun, is_open = (d[c].values for c in ("t_out", "heater_kw", "solar_wm2", "open"))
    y = d.t_in.values
    twin, pred = np.zeros(len(d)), np.zeros(len(d))
    twin[0] = pred[0] = y[0]
    for k in range(len(d) - 1):
        p = twin[k] + DT / C * ((t_out[k] - twin[k]) / R + heat[k] + g * sun[k] + q * is_open[k])
        pred[k + 1] = p
        twin[k + 1] = p + gain * (y[k + 1] - p)
    return twin, pred

def rmse(a, b):
    return round(float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float)) ** 2))), 3)

print(WEEK1[["t_in", "t_out", "heater_kw", "solar_wm2", "open"]].describe().round(2))
first = simulate(WEEK1, **FIRST)[0]
print(f"heater energy in week 1: {WEEK1.heater_kw.sum() * DT:.0f} kWh")
print(f"first twin, week 1: RMSE {rmse(first, WEEK1.t_in)} K, mean error {float((first - WEEK1.t_in).mean()):+.2f} K")

**Python note.** The function `simulate` loops over the 672 entries of a week. Each step uses the inputs of its own quarter of an hour, as the logger records them, and the twin starts from the first measurement.

**Exercise A.** Store the RMSE in K of the first twin in week 2 in `r_a`.

In [ ]:
r_a = None   # your answer

check("Exercise A", r_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_a = rmse(simulate(WEEK2, **FIRST)[0], WEEK2.t_in)

check("Exercise A", r_a)

In [ ]:
# Figure: the first week of the library with the first twin and the heater
fig, ax = plt.subplots(2, 1, figsize=(10, 4.8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax[0].plot(WEEK1.index, WEEK1.t_in, ".", ms=2, color="#1B2430", label="measured")
ax[0].plot(WEEK1.index, first, color="#C98510", lw=1.2, label="first twin")
ax[0].set_ylabel("indoor temperature (C)")
ax[0].legend(fontsize=8, loc="lower right")
ax[1].fill_between(WEEK1.index, WEEK1.heater_kw, step="post", color="#B53F2E", alpha=0.5, label="heater")
ax[1].plot(WEEK1.index, WEEK1.t_out, color="#1E6FA8", lw=1, label="outdoor temperature")
ax[1].set_ylabel("kW and C")
ax[1].legend(fontsize=8, loc="upper right")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout()
plt.show()

**Observe and reflect.** The first twin runs too cold. Which heat flows does it leave out, and at which hours does it miss the library most?

## 2. Calibration: Your twin in the reading room &#91;[7](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-7)&#93;

*Lecture: [Calibration: Fitting the twin to the library](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#calibration)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "simulate" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A twin is calibrated when its parameters reproduce the measured behaviour. Python searches the heat capacity, the resistance, the solar aperture and the internal gains that bring the simulated week 1 closest to the measured one. It then tests the twin on week 2, which the calibration has not seen.

**From the formulas to the code.** Output-error least squares chooses $\theta = (C, R, g, q)$ to minimise $\sum_k (\hat{T}_k(\theta) - y_k)^2$, where $\hat{T}_k$ is the open-loop twin of the whole week: `calibrate(d)`. The time constant of the calibrated room is $\tau = R\,C$.

In [ ]:
def calibrate(d, x0=(25.0, 0.5, 0.0, 0.0)):
    """Output-error calibration: the C, R, g and q that bring the open-loop twin closest to the measurements of d."""
    res = least_squares(lambda p: simulate(d, *p)[0] - d.t_in.values, x0=x0)
    return dict(zip(("C", "R", "g", "q"), (float(v) for v in res.x)))

FIT = calibrate(WEEK1)
print("calibrated:", {k: round(v, 4) for k, v in FIT.items()}, "| time constant R C =", round(FIT["R"] * FIT["C"], 2), "h")
for name, pars in (("first", FIRST), ("calibrated", FIT)):
    print(f"{name:>10} twin: RMSE week 1 {rmse(simulate(WEEK1, **pars)[0], WEEK1.t_in)} K, "
          f"week 2 {rmse(simulate(WEEK2, **pars)[0], WEEK2.t_in)} K")

**Python note.** The expression `**pars` unpacks a dictionary into named arguments, so `simulate(WEEK2, **FIT)` passes C, R, g and q. The lambda inside `calibrate` returns one error per entry, 672 in all, and `least_squares` minimises the sum of their squares.

**Exercise B.** Store the RMSE in K of the calibrated twin in week 2 in `r_b`.

In [ ]:
r_b = None   # your answer

check("Exercise B", r_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_b = rmse(simulate(WEEK2, **FIT)[0], WEEK2.t_in)

check("Exercise B", r_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lab.html) opens the reading room of the library with its twin. Paste the copied rows into `MY_TWINS`, and the code runs each of your twins again on the same week.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_TWINS = """
"""
if MY_TWINS.strip():
    runs = pd.read_csv(io.StringIO(MY_TWINS.strip()))
    source = "your own twins in the reading room"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4], "week": [1, 1, 1, 2], "C": [30.0, 30.0, 31.5, 31.5],
                         "R": [0.4, 0.45, 0.441, 0.441], "q": [0.0, 7.0, 7.2, 7.2], "g": [0.0, 0.02, 0.018, 0.018],
                         "rmse": [3.395, 0.418, 0.278, 0.284]})
    source = "a simulated student"
WEEKS = {1: WEEK1, 2: WEEK2}
runs["model_rmse"] = [rmse(simulate(WEEKS[r.week], r.C, r.R, r.g, r.q)[0], WEEKS[r.week].t_in) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "week", "C", "R", "q", "g", "rmse", "model_rmse"]].to_string(index=False))

In [ ]:
# Figure: the first twin and the calibrated twin in week 1 and week 2
both = pd.concat([WEEK1, WEEK2])
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(both.index, both.t_in, ".", ms=1.5, color="#1B2430", label="measured")
for name, pars, col in (("first twin", FIRST, "#C98510"), ("calibrated twin", FIT, "#1E6FA8")):
    twin = np.r_[simulate(WEEK1, **pars)[0], simulate(WEEK2, **pars)[0]]
    ax.plot(both.index, twin, color=col, lw=1, label=name)
ax.axvline(WEEK2.index[0], color="#56616D", ls=":")
ax.text(WEEK2.index[0], ax.get_ylim()[1] - 0.6, " week 2: validation", fontsize=8)
ax.set_ylabel("indoor temperature (C)")
ax.legend(fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Why must the twin be tested on a week that the calibration has not seen? Which parameter would you trust least, and why?

## 3. Synchronisation: The twin that follows &#91;[9](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-9)&#93;

*Lecture: [Synchronisation: Keeping the twin in step](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#synchronisation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "FIT" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An open-loop twin drifts away from the library as its small errors add up. A synchronised twin corrects its state with every measurement. Python tries several gains of the correction and compares the twin with the true temperature, which the simulation of the library knows. It then shows what wrong time stamps of the heater meter do to the calibration.

**From the formulas to the code.** Each step predicts $\hat{T}^-_{k+1} = f(\hat{T}_k, u_k)$ with the model and then corrects $\hat{T}_{k+1} = \hat{T}^-_{k+1} + L\,(y_{k+1} - \hat{T}^-_{k+1})$ with the gain $L$: `simulate(..., gain=L)`. The difference $y_{k+1} - \hat{T}^-_{k+1}$ is the innovation. The function `late(d, steps)` shifts the heater column by whole quarters of an hour.

In [ ]:
print("sensor alone: RMSE against the true temperature", rmse(WEEK2.t_in, WEEK2.t_true), "K")
for gain in (0.0, 0.1, 0.2, 0.3, 0.5, 1.0):
    twin, pred = simulate(WEEK2, **FIT, gain=gain)
    print(f"gain {gain:.1f}: twin against the true temperature {rmse(twin, WEEK2.t_true)} K | "
          f"spread of the innovations {float(np.std(WEEK2.t_in.values[1:] - pred[1:])):.3f} K")

def late(d, steps):
    """The data with the heater meter shifted by steps quarters of an hour (positive: its values are stamped too late)."""
    d = d.copy()
    d["heater_kw"] = np.roll(d.heater_kw.values, steps)
    return d

for steps in (-2, -1, 0, 1, 2):
    p = calibrate(late(WEEK1, steps))
    print(f"heater stamps {15 * steps:+4d} min: RMSE of the calibrated twin {rmse(simulate(late(WEEK1, steps), **p)[0], WEEK1.t_in)} K,"
          f" q = {p['q']:.2f} kW, R = {p['R']:.3f} K/kW")

**Python note.** The function `np.roll` moves the whole column by `steps` rows and wraps the end around to the start. For a shift of a few rows in a week this end effect is negligible.

**Exercise C.** Store the RMSE in K of the synchronised twin with the gain 0.2 against the true temperature in week 2 in `r_c`.

In [ ]:
r_c = None   # your answer

check("Exercise C", r_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_c = rmse(simulate(WEEK2, **FIT, gain=0.2)[0], WEEK2.t_true)

check("Exercise C", r_c)

In [ ]:
# Figure: the error of the twin against the gain, and one day with the measurement and the twins
gains = np.round(np.arange(0, 1.01, 0.05), 2)
errors = [rmse(simulate(WEEK2, **FIT, gain=L)[0], WEEK2.t_true) for L in gains]
day = WEEK2.loc["2026-02-11"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(gains, errors, "o-", color="#1E6FA8", ms=3)
ax[0].axhline(rmse(WEEK2.t_in, WEEK2.t_true), color="#B53F2E", ls="--", lw=1, label="sensor alone")
ax[0].set_xlabel("gain L of the synchronisation")
ax[0].set_ylabel("RMSE against the true temperature (K)")
ax[0].set_ylim(0, 0.3)
ax[0].legend(fontsize=8)
sl = slice(WEEK2.index.get_loc(day.index[0]), WEEK2.index.get_loc(day.index[-1]) + 1)
ax[1].plot(day.index, day.t_in, ".", ms=3, color="#1B2430", label="measured")
ax[1].plot(day.index, day.t_true, color="#56616D", ls="--", lw=1, label="true")
ax[1].plot(day.index, simulate(WEEK2, **FIT)[0][sl], color="#C98510", lw=1, label="open-loop twin")
ax[1].plot(day.index, simulate(WEEK2, **FIT, gain=0.3)[0][sl], color="#1E6FA8", lw=1.4, label="synchronised, L = 0.3")
ax[1].set_ylabel("indoor temperature (C)")
ax[1].legend(fontsize=7, loc="lower right")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Why can the synchronised twin know the temperature better than its own sensor? What goes wrong with a gain of 1?

## 4. Drift and recalibration: The open window &#91;[11](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#ref-11)&#93;

*Lecture: [Drift and recalibration](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#recalibration)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "late" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** In week 3 a window of the reading room stays open from Wednesday 10:00 to Thursday 16:00. Python watches the innovations of the synchronised twin with the cumulative sum (CUSUM) of Week 10. It also recalibrates the resistance R every hour from a sliding window of the last hours, and compares it with the true resistance of the library with the window open.

**From the formulas to the code.** The innovations are standardised with their spread in week 2, $z_k = (y_k - \hat{T}^-_k)/\sigma$. The CUSUM alarms when one of its sums leaves $\pm h$, with $\kappa = 0.5$ and $h = 5$: `watch(gain)`. `recalibrate_R(d)` finds the R of the open-loop twin over the data d by a golden-section search, with C, g and q kept at the calibration. The envelope and the open window in parallel give $R = 1/(1/R_{env} + 1/R_{window})$.

In [ ]:
def cusum(z, kappa=0.5, h=5.0):
    """The CUSUM of Week 10; returns the positions of its alarms and starts again at zero after each one."""
    sp = sm = 0.0
    alarms = []
    for k, v in enumerate(z):
        sp, sm = max(0.0, sp + v - kappa), min(0.0, sm + v + kappa)
        if sp > h or sm < -h:
            alarms.append(k)
            sp = sm = 0.0
    return alarms

def watch(gain):
    """The synchronised twin over week 3 and the CUSUM on its standardised innovations (spread from week 2)."""
    sigma = float(np.std(WEEK2.t_in.values[1:] - simulate(WEEK2, **FIT, gain=gain)[1][1:]))
    twin, pred = simulate(WEEK3, **FIT, gain=gain)
    times = WEEK3.index[cusum(np.r_[0.0, (WEEK3.t_in.values[1:] - pred[1:]) / sigma])]
    first = next((t for t in times if t >= OPEN_FROM), None)
    return dict(twin=twin, alarms=times, false=int((times < OPEN_FROM).sum()),
                first_min=None if first is None else int((first - OPEN_FROM) / pd.Timedelta(minutes=1)),
                rmse_true=rmse(twin, WEEK3.t_true))

def recalibrate_R(d):
    """Golden-section search for the R that brings the open-loop twin closest to d; C, g and q stay at FIT."""
    err = lambda r: float(np.mean((simulate(d, FIT["C"], r, FIT["g"], FIT["q"])[0] - d.t_in.values) ** 2))
    a, b, ratio = 0.1, 1.2, (math.sqrt(5) - 1) / 2
    c, e = b - ratio * (b - a), a + ratio * (b - a)
    fc, fe = err(c), err(e)
    for _ in range(40):
        if fc < fe:
            b, e, fe = e, c, fc
            c = b - ratio * (b - a)
            fc = err(c)
        else:
            a, c, fc = c, e, fe
            e = a + ratio * (b - a)
            fe = err(e)
    return (a + b) / 2

R_NORMAL = 1 / (1 / (R_AIR_MASS + R_MASS_OUT) + 1 / R_GAPS)
R_OPEN = 1 / (1 / (R_AIR_MASS + R_MASS_OUT) + 1 / R_WINDOW)
print(f"true R of the library: {R_NORMAL:.3f} K/kW, with the window open {R_OPEN:.3f} K/kW")
for gain in (0.0, 0.1, 0.3, 0.5, 1.0):
    w = watch(gain)
    print(f"gain {gain:.1f}: false alarms before the opening {w['false']}, first alarm {w['first_min']} min after it,"
          f" RMSE against the true temperature {w['rmse_true']} K")
TRACKS = {}
for hours in (12, 24, 48):
    times = pd.date_range(WEEK3.index[0] + pd.Timedelta(hours=hours), WEEK3.index[-1], freq="1h")
    TRACKS[hours] = pd.Series([recalibrate_R(WEEK3.loc[t - pd.Timedelta(hours=hours):t]) for t in times], index=times)
    print(f"window {hours:>2} h: R at Thu 08:00 {TRACKS[hours]['2026-02-19 08:00']:.3f}, at Sat 00:00 "
          f"{TRACKS[hours]['2026-02-21 00:00']:.3f} K/kW")

**Python note.** The golden-section search keeps an interval around the minimum and shrinks it by the factor 0.618 in each step. After 40 steps the interval, 1.1 K/kW wide at the start, is narrower than 0.00000001 K/kW. The interactive lab uses the same search, so both give the same R.

**Exercise D.** Store the R in K/kW recalibrated at Thursday 08:00 with a window of 24 h, rounded to three decimals, in `r_d`.

In [ ]:
r_d = None   # your answer

check("Exercise D", r_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
r_d = round(TRACKS[24]["2026-02-19 08:00"], 3)

check("Exercise D", r_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lab.html) runs the twin of week 3 on the server of the campus. Paste the copied rows into `MY_RUNS`, and the code repeats each of your runs.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs on the twin server"
else:
    runs = pd.DataFrame({"run": [1, 2, 3], "gain": [0.0, 0.3, 0.3], "window_h": [24, 24, 12],
                         "false_alarms": [7, 0, 0], "first_alarm_min": [90, 45, 45], "r_thu08": [0.293, 0.293, 0.286]})
    source = "a simulated student"
runs["model_false_alarms"] = [watch(r.gain)["false"] for r in runs.itertuples()]
runs["model_first_alarm_min"] = [watch(r.gain)["first_min"] for r in runs.itertuples()]
runs["model_r_thu08"] = [round(TRACKS[int(r.window_h)]["2026-02-19 08:00"], 3) for r in runs.itertuples()]
print("Data:", source)
print(runs[["run", "gain", "window_h", "false_alarms", "model_false_alarms", "first_alarm_min", "model_first_alarm_min",
            "r_thu08", "model_r_thu08"]].to_string(index=False))

In [ ]:
# Figure: week 3 with the open window, the alarms of the twin and the recalibrated resistance
w = watch(0.3)
fig, ax = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
for a in ax:
    a.axvspan(OPEN_FROM, OPEN_TO, color="#F2B33D", alpha=0.25)
ax[0].plot(WEEK3.index, WEEK3.t_in, ".", ms=1.5, color="#1B2430", label="measured")
ax[0].plot(WEEK3.index, w["twin"], color="#1E6FA8", lw=1, label="synchronised twin, L = 0.3")
ax[0].plot(w["alarms"], [16.0] * len(w["alarms"]), "|", ms=10, color="#B53F2E", label="alarms of the CUSUM")
ax[0].set_ylabel("indoor temperature (C)")
ax[0].set_ylim(15.5, 23.6)
ax[0].legend(fontsize=8, loc="upper center", ncol=3)
for hours, col in ((12, "#2E8B57"), (24, "#1E6FA8"), (48, "#C98510")):
    ax[1].plot(TRACKS[hours].index, TRACKS[hours], color=col, lw=1.2, label=f"window {hours} h")
ax[1].axhline(R_NORMAL, color="#56616D", ls=":", lw=1)
ax[1].axhline(R_OPEN, color="#B53F2E", ls=":", lw=1)
ax[1].set_ylabel("recalibrated R (K/kW)")
ax[1].legend(fontsize=8, loc="lower right")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout()
plt.show()

**Observe and reflect.** Why does the short window find the new R first, and what does it cost when the window is closed again?

## 5. The campus microgrid, stage 12

*Lecture: [The campus microgrid, stage 12: A twin that follows](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#the-campus-microgrid-stage-12)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "TRACKS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 12 gives the campus its first digital twin. It is calibrated on one week, tested on another, synchronised with every measurement and watched for drift. The numbers come from Sections 1 to 4.

In [ ]:
w3 = watch(0.3)
synced = simulate(WEEK2, **FIT, gain=0.3)[0]
STAGE12 = {
    "first twin: RMSE in week 1 (K)": rmse(simulate(WEEK1, **FIRST)[0], WEEK1.t_in),
    "calibrated twin: C (kWh/K)": round(FIT["C"], 2),
    "calibrated twin: R (K/kW)": round(FIT["R"], 3),
    "calibrated twin: internal gains q (kW)": round(FIT["q"], 2),
    "calibrated twin: RMSE in week 1 (K)": rmse(simulate(WEEK1, **FIT)[0], WEEK1.t_in),
    "calibrated twin: RMSE in week 2 (K)": rmse(simulate(WEEK2, **FIT)[0], WEEK2.t_in),
    "sensor alone: RMSE against the true temperature (K)": rmse(WEEK2.t_in, WEEK2.t_true),
    "synchronised twin, L = 0.3: RMSE against the true temperature (K)": rmse(synced, WEEK2.t_true),
    "open window: false alarms before the opening": w3["false"],
    "open window: first alarm after the opening (min)": w3["first_min"],
    "open window: R recalibrated at Thu 08:00, window 24 h (K/kW)": round(TRACKS[24]["2026-02-19 08:00"], 3),
    "open window: true R (K/kW)": round(R_OPEN, 3),
    "heater stamps 30 min late: RMSE of the calibrated twin (K)": rmse(simulate(late(WEEK1, 2), **calibrate(late(WEEK1, 2)))[0],
                                                                       WEEK1.t_in),
}
for key, value in STAGE12.items():
    print(f"{key:<68} {value}")

**Python note.** Every value of `STAGE12` is computed again from the functions of the earlier sections, so the table changes with them if a section is changed.

In [ ]:
# Figure: the errors of the twins and the resistance recalibrated around the open window
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.3]})
names = ["first twin", "calibrated twin", "sensor alone", "synchronised twin"]
values = [rmse(simulate(WEEK2, **FIRST)[0], WEEK2.t_in), STAGE12["calibrated twin: RMSE in week 2 (K)"],
          STAGE12["sensor alone: RMSE against the true temperature (K)"],
          STAGE12["synchronised twin, L = 0.3: RMSE against the true temperature (K)"]]
ax[0].barh(names[::-1], values[::-1], color=["#1E6FA8", "#56616D", "#9DB7CC", "#C98510"])
ax[0].set_xscale("log")
ax[0].set_xlabel("RMSE in week 2 (K, logarithmic)")
ax[1].plot(TRACKS[24].index, TRACKS[24], color="#1E6FA8", lw=1.4, label="recalibrated R, window 24 h")
ax[1].axvspan(OPEN_FROM, OPEN_TO, color="#F2B33D", alpha=0.25)
ax[1].axhline(R_NORMAL, color="#56616D", ls=":", lw=1, label="true R, windows closed")
ax[1].axhline(R_OPEN, color="#B53F2E", ls=":", lw=1, label="true R, window open")
ax[1].set_ylabel("R (K/kW)")
ax[1].legend(fontsize=8, loc="lower left")
ax[1].xaxis.set_major_formatter(mdates.DateFormatter("%a %d"))
plt.tight_layout()
plt.show()

**Exercise E.** Store the minutes from the opening of the window to the first alarm of the twin, as kept in `STAGE12`, in `m_e`.

In [ ]:
m_e = None   # your answer

check("Exercise E", m_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
m_e = STAGE12["open window: first alarm after the opening (min)"]

check("Exercise E", m_e)

**Observe and reflect.** Which number of stage 12 would you show the facility manager to argue for a twin of every building of the campus?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE12" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-12/"
               "NB12_digital_twin_sync_calibration.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Wherever a quantity rises and falls like the temperature of a room, a first-order twin can follow it. The function of this section simulates six days of a system of your choice, calibrates its twin on four days, tests it on two and synchronises it.

The systems and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "cold_store"
# choose: cold_store, aquarium, server_room, greenhouse, hot_water_tank, battery_pack
APPS = {  # description, unit of the state, C, R, power of the actuator, its on and off band, ambient and its swing, gain
    "cold_store": ("a cold store for vegetables", "C", 8.0, 1.5, -12.0, (4.5, 3.5), (18.0, 6.0), 0.0),
    "aquarium": ("a public aquarium tank with its heater", "C", 3.0, 4.0, 3.0, (25.5, 26.5), (20.0, 2.0), 0.0),
    "server_room": ("a server room with 20 kW of servers and its cooling", "C", 12.0, 0.8, -30.0, (24.5, 22.5),
                    (22.0, 5.0), 20.0),
    "greenhouse": ("a greenhouse with a gas heater", "C", 2.5, 0.2, 60.0, (15.5, 16.5), (8.0, 6.0), 0.0),
    "hot_water_tank": ("a hot water tank of 1000 litres in a sports hall", "C", 1.16, 50.0, 6.0, (55.0, 60.0),
                       (20.0, 1.0), -2.0),
    "battery_pack": ("the cooling of the battery container of the campus", "C", 4.0, 1.0, -25.0, (36.0, 32.0),
                     (25.0, 4.0), 20.0),
}

def app_twin(app, seed=7):
    desc, unit, C, R, power, band, (amb0, swing), gain = APPS[app]
    rng = np.random.default_rng(seed)
    n, dt = 6 * 24 * 60, 1 / 60                                    # six days, minute by minute
    hour = np.arange(n) * dt % 24
    amb = amb0 + swing * np.sin(2 * np.pi * (hour - 9) / 24) + np.convolve(rng.normal(0, 1, n), np.ones(120) / 120, "same")
    extra = gain * (1 + 0.05 * rng.normal(0, 1, n))                 # a steady flow: servers, losses or draws
    x = np.zeros(n)
    u = np.zeros(n)
    x[0], on = np.mean(band), False
    lo, hi = min(band), max(band)
    for k in range(n - 1):
        if power > 0:
            on = True if x[k] < lo else (False if x[k] > hi else on)
        else:
            on = True if x[k] > hi else (False if x[k] < lo else on)
        u[k] = power if on else 0.0
        x[k + 1] = x[k] + dt / C * ((amb[k] - x[k]) / R + u[k] + extra[k])
    u[-1] = u[-2]
    df = pd.DataFrame({"t_true": x[::15], "t_in": x[::15] + rng.normal(0, 0.05, n // 15), "t_out": amb.reshape(-1, 15).mean(1),
                       "heater_kw": u.reshape(-1, 15).mean(1), "solar_wm2": 0.0, "open": 1.0})
    train, test = df.iloc[:4 * 96], df.iloc[4 * 96:]
    res = least_squares(lambda p: simulate(train, p[0], p[1], 0.0, p[2])[0] - train.t_in.values,
                        x0=[1.5 * C, 0.7 * R, 0.0], bounds=([1e-3, 1e-3, -1e3], [1e3, 1e3, 1e3]))
    Cf, Rf, qf = res.x
    print(f"{app}: {desc}")
    print(f"  true C = {C}, R = {R}; calibrated C = {Cf:.3f}, R = {Rf:.3f}, constant flow {qf:+.2f}")
    print(f"  open-loop twin on the last two days: RMSE {rmse(simulate(test, Cf, Rf, 0.0, qf)[0], test.t_in)} {unit}")
    print(f"  synchronised twin, L = 0.3: RMSE against the true state "
          f"{rmse(simulate(test, Cf, Rf, 0.0, qf, gain=0.3)[0], test.t_true)} {unit} (sensor alone "
          f"{rmse(test.t_in, test.t_true)} {unit})")

app_twin(APP)

**Python note.** The function `app_twin` reuses `simulate` from Section 1. Its columns `t_out`, `heater_kw` and `open` now carry the ambient temperature, the actuator and a constant 1. The model learns the steady extra flow as `q`.

**Observe and reflect.** For which system does the twin learn its parameters best, and why? What would you log every 15 minutes to build the twin of your own system?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Turn `simulate`, `calibrate` and `recalibrate_R` into the methods of one class `LibraryTwin`, and test that it reproduces the numbers of stage 12. |
| Embedded systems | The thermostat of the library could run the twin itself. Count the operations of one step and estimate the energy of a year of steps on a microcontroller. |
| Internet of Things | Lose 10 percent of the measurements at random and let the twin skip the correction for them. How much worse does it follow the library? |
| Computer networks | The measurements arrive 2 to 6 minutes late at the twin server. Extend `simulate` so that it corrects the state with a delayed measurement. |
| Control engineering | Use the calibrated twin to start the heater in the morning so that the library reaches 21 C at 8:00, and compare the energy with the thermostat. |
| Data science | Calibrate a twin with two heat capacities, one for the air and one for the walls, and test whether it beats the twin of this week on week 2. |
| Cybersecurity | An attacker adds 0.5 K to the measurements of week 3 from Friday on. Which part of the twin notices it first, the CUSUM or the recalibrated R? |
| Artificial intelligence | Train a small neural network to predict the next temperature from the last four and the inputs, and compare its open-loop week with the calibrated twin. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-12/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-12).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>